In [ ]:
#@title Install Dependencies
%%capture
!pip install gradio -q

In [ ]:
#@title Evaluation Suite Builder App
"""
Evaluation Suite Builder - Design custom evaluation rubrics
Helps PMs define what 'good' looks like for their AI use case
"""

import gradio as gr
import json

USE_CASE_TEMPLATES = {
    "Customer Service Bot": {
        "dimensions": ["Helpfulness", "Tone", "Accuracy", "Completeness"],
        "weights": [0.3, 0.25, 0.25, 0.2],
        "description": "Evaluates support responses for customer satisfaction and issue resolution"
    },
    "Content Generator": {
        "dimensions": ["Creativity", "Relevance", "Clarity", "Engagement"],
        "weights": [0.3, 0.25, 0.25, 0.2],
        "description": "Evaluates marketing content, blog posts, and creative writing"
    },
    "Code Assistant": {
        "dimensions": ["Accuracy", "Completeness", "Clarity", "Best Practices"],
        "weights": [0.35, 0.25, 0.2, 0.2],
        "description": "Evaluates code suggestions, explanations, and debugging help"
    },
    "Research Analyst": {
        "dimensions": ["Accuracy", "Completeness", "Clarity", "Source Quality"],
        "weights": [0.35, 0.25, 0.2, 0.2],
        "description": "Evaluates summaries, analysis, and research synthesis"
    }
}

DIMENSION_DEFINITIONS = {
    "Accuracy": {
        "description": "Information is factually correct and verifiable",
        "rubric": {
            "5": "All facts verifiable, no errors",
            "4": "Minor inaccuracies that don't affect core message",
            "3": "Some factual issues but main points correct",
            "2": "Significant errors affecting reliability",
            "1": "Mostly incorrect or misleading"
        }
    },
    "Helpfulness": {
        "description": "Response provides actionable value to the user",
        "rubric": {
            "5": "Fully addresses need with clear next steps",
            "4": "Addresses need, minor gaps in guidance",
            "3": "Partially helpful, some key info missing",
            "2": "Limited value, doesn't address core need",
            "1": "Not helpful, off-topic or confusing"
        }
    },
    "Clarity": {
        "description": "Response is easy to understand and well-structured",
        "rubric": {
            "5": "Crystal clear, perfectly organized",
            "4": "Clear with good structure",
            "3": "Understandable but could be clearer",
            "2": "Confusing or poorly organized",
            "1": "Very difficult to understand"
        }
    },
    "Tone": {
        "description": "Tone is appropriate for context and audience",
        "rubric": {
            "5": "Perfect tone for situation",
            "4": "Appropriate with minor adjustments needed",
            "3": "Acceptable but not optimal",
            "2": "Noticeably off for the context",
            "1": "Completely inappropriate"
        }
    },
    "Completeness": {
        "description": "Response covers all necessary aspects",
        "rubric": {
            "5": "Comprehensive, nothing missing",
            "4": "Covers most important points",
            "3": "Addresses basics, misses some aspects",
            "2": "Significant gaps",
            "1": "Very incomplete"
        }
    },
    "Creativity": {
        "description": "Response shows originality and creative thinking",
        "rubric": {
            "5": "Highly original and innovative",
            "4": "Good creative elements",
            "3": "Some originality",
            "2": "Mostly generic",
            "1": "Completely formulaic"
        }
    },
    "Relevance": {
        "description": "Response directly addresses the prompt/question",
        "rubric": {
            "5": "Perfectly on-topic throughout",
            "4": "Mostly relevant with minor tangents",
            "3": "Addresses topic but wanders",
            "2": "Partially off-topic",
            "1": "Doesn't address the question"
        }
    },
    "Engagement": {
        "description": "Content captures and holds reader attention",
        "rubric": {
            "5": "Compelling, want to read more",
            "4": "Interesting and engaging",
            "3": "Readable but not exciting",
            "2": "Dry or boring",
            "1": "Tedious, hard to finish"
        }
    },
    "Best Practices": {
        "description": "Follows established standards and conventions",
        "rubric": {
            "5": "Exemplary adherence to standards",
            "4": "Follows most best practices",
            "3": "Some deviations from standards",
            "2": "Significant issues with practices",
            "1": "Ignores established standards"
        }
    },
    "Source Quality": {
        "description": "Information comes from reliable, credible sources",
        "rubric": {
            "5": "All claims well-sourced",
            "4": "Most information traceable",
            "3": "Some sourcing, some unsupported",
            "2": "Poor sourcing",
            "1": "No credible sources"
        }
    }
}


def load_template(use_case):
    """Load a template configuration"""
    if use_case in USE_CASE_TEMPLATES:
        template = USE_CASE_TEMPLATES[use_case]
        dims = template["dimensions"]
        weights = template["weights"]
        return (
            dims[0] if len(dims) > 0 else "Accuracy",
            weights[0] if len(weights) > 0 else 0.25,
            dims[1] if len(dims) > 1 else "Clarity",
            weights[1] if len(weights) > 1 else 0.25,
            dims[2] if len(dims) > 2 else "Helpfulness",
            weights[2] if len(weights) > 2 else 0.25,
            dims[3] if len(dims) > 3 else "Completeness",
            weights[3] if len(weights) > 3 else 0.25,
            template["description"]
        )
    return "Accuracy", 0.25, "Clarity", 0.25, "Helpfulness", 0.25, "Completeness", 0.25, ""


def show_rubric(dimension):
    """Display the rubric for a dimension"""
    if dimension in DIMENSION_DEFINITIONS:
        info = DIMENSION_DEFINITIONS[dimension]
        output = f"## {dimension}\n\n"
        output += f"**Definition:** {info['description']}\n\n"
        output += "### Scoring Rubric\n\n"
        for score, desc in info['rubric'].items():
            output += f"- **{score}:** {desc}\n"
        return output
    return "Select a dimension to see its rubric."


def generate_suite(dim1, w1, dim2, w2, dim3, w3, dim4, w4, suite_name, description):
    """Generate the evaluation suite configuration"""
    
    total_weight = w1 + w2 + w3 + w4
    if abs(total_weight - 1.0) > 0.01:
        return f"Weights must sum to 1.0 (currently: {total_weight:.2f})", ""
    
    suite = {
        "name": suite_name or "Custom Evaluation Suite",
        "description": description or "Custom evaluation configuration",
        "dimensions": [
            {
                "name": dim1,
                "weight": w1,
                "definition": DIMENSION_DEFINITIONS.get(dim1, {}).get("description", ""),
                "rubric": DIMENSION_DEFINITIONS.get(dim1, {}).get("rubric", {})
            },
            {
                "name": dim2,
                "weight": w2,
                "definition": DIMENSION_DEFINITIONS.get(dim2, {}).get("description", ""),
                "rubric": DIMENSION_DEFINITIONS.get(dim2, {}).get("rubric", {})
            },
            {
                "name": dim3,
                "weight": w3,
                "definition": DIMENSION_DEFINITIONS.get(dim3, {}).get("description", ""),
                "rubric": DIMENSION_DEFINITIONS.get(dim3, {}).get("rubric", {})
            },
            {
                "name": dim4,
                "weight": w4,
                "definition": DIMENSION_DEFINITIONS.get(dim4, {}).get("description", ""),
                "rubric": DIMENSION_DEFINITIONS.get(dim4, {}).get("rubric", {})
            }
        ],
        "scoring": {
            "scale": "1-5",
            "method": "weighted_average"
        }
    }
    
    # Summary
    summary = f"""## {suite['name']}

{suite['description']}

### Evaluation Dimensions

| Dimension | Weight | Definition |
|-----------|--------|------------|
| {dim1} | {w1*100:.0f}% | {DIMENSION_DEFINITIONS.get(dim1, {}).get('description', '')} |
| {dim2} | {w2*100:.0f}% | {DIMENSION_DEFINITIONS.get(dim2, {}).get('description', '')} |
| {dim3} | {w3*100:.0f}% | {DIMENSION_DEFINITIONS.get(dim3, {}).get('description', '')} |
| {dim4} | {w4*100:.0f}% | {DIMENSION_DEFINITIONS.get(dim4, {}).get('description', '')} |

### Scoring Method
- Scale: 1-5 points per dimension
- Final Score: Weighted average
- Minimum passing: 3.0 (configurable)
"""
    
    json_output = json.dumps(suite, indent=2)
    
    return summary, json_output


# Build Gradio interface
all_dimensions = list(DIMENSION_DEFINITIONS.keys())

with gr.Blocks(title="Evaluation Suite Builder", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # Evaluation Suite Builder
    
    Design custom evaluation rubrics for your AI use case.
    
    **For Product Managers:** Define what 'good' looks like before you build.
    """)
    
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Configuration")
            
            template_dropdown = gr.Dropdown(
                choices=list(USE_CASE_TEMPLATES.keys()),
                label="Load Template",
                value="Customer Service Bot"
            )
            
            suite_name = gr.Textbox(label="Suite Name", value="My Evaluation Suite")
            description = gr.Textbox(label="Description", lines=2)
            
            gr.Markdown("### Dimensions & Weights")
            gr.Markdown("*Weights must sum to 1.0*")
            
            with gr.Row():
                dim1 = gr.Dropdown(choices=all_dimensions, value="Helpfulness", label="Dimension 1")
                w1 = gr.Slider(0, 0.5, value=0.3, step=0.05, label="Weight")
            
            with gr.Row():
                dim2 = gr.Dropdown(choices=all_dimensions, value="Tone", label="Dimension 2")
                w2 = gr.Slider(0, 0.5, value=0.25, step=0.05, label="Weight")
            
            with gr.Row():
                dim3 = gr.Dropdown(choices=all_dimensions, value="Accuracy", label="Dimension 3")
                w3 = gr.Slider(0, 0.5, value=0.25, step=0.05, label="Weight")
            
            with gr.Row():
                dim4 = gr.Dropdown(choices=all_dimensions, value="Completeness", label="Dimension 4")
                w4 = gr.Slider(0, 0.5, value=0.2, step=0.05, label="Weight")
            
            generate_btn = gr.Button("Generate Suite", variant="primary")
        
        with gr.Column(scale=1):
            gr.Markdown("### Rubric Preview")
            rubric_preview = gr.Markdown()
            
            gr.Markdown("### Generated Suite")
            suite_summary = gr.Markdown()
        
        with gr.Column(scale=1):
            gr.Markdown("### Export (JSON)")
            json_output = gr.Code(language="json")
    
    # Event handlers
    template_dropdown.change(
        fn=load_template,
        inputs=[template_dropdown],
        outputs=[dim1, w1, dim2, w2, dim3, w3, dim4, w4, description]
    )
    
    for dim in [dim1, dim2, dim3, dim4]:
        dim.change(fn=show_rubric, inputs=[dim], outputs=[rubric_preview])
    
    generate_btn.click(
        fn=generate_suite,
        inputs=[dim1, w1, dim2, w2, dim3, w3, dim4, w4, suite_name, description],
        outputs=[suite_summary, json_output]
    )
    
    demo.load(fn=lambda: show_rubric("Helpfulness"), outputs=[rubric_preview])

In [ ]:
#@title Launch App - Copy the gradio.live URL below
demo.launch(share=True)